# ARC2 014 V 27B service probe
Single vLLM server (TP4/64K/C4/FP8, KV default, no MTP), hello xhigh/low short protocol, then P2 (291dc1e1, db118e2a) if budget remains. Diagnostic only; no score claimed. Known negative outcomes exit 0 with explicit status in run.json.

In [ ]:
import base64, hashlib, json, subprocess, sys, time
from pathlib import Path
notebook_started = time.time()
source_dir = Path('/kaggle/working/014-v-probe-code')
source_dir.mkdir(exist_ok=False)
blobs = json.loads(base64.b64decode('...'))
expected = {
  "vdriver.py": "7a8c49be73d41d33c578cccbe89f2250840b8cb61d2265473fedb49ec92dd0cd",
  "run_reasoning_probe.py": "827934ec2865102eea31493acc2bd230da2fa1ebc2315cfeaf558fdde561e1c9",
  "p2-prompts.json": "9f0c23dc53994fc21a937ebfc1ea51d11af4377281ed07fa44bdcb723854dd83"
}
for name, encoded in blobs.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == expected[name], name
    (source_dir / name).write_bytes(raw)
print(json.dumps({'event': 'boot', 'source_sha256': expected,
                   'python': sys.version, 'notebook_started': notebook_started}), flush=True)
result = subprocess.run([sys.executable, '-u', str(source_dir / 'vdriver.py'),
                         '--work', '/kaggle/working/vprobe',
                         '--prompts', str(source_dir / 'p2-prompts.json'),
                         '--session-seconds', '3300'], cwd=str(source_dir))
print('VDRIVER_RETURN_CODE', result.returncode, flush=True)
assert result.returncode == 0, 'Inspect /kaggle/working/vprobe/run.json'
